# Flight price prediction: linear regression pipeline
Simplified version of `flight_lr.ipynb`. Run top to bottom from the project folder in `amazon_ml`.

The original calculations are preserved: three segments, four Ridge variants, fare statistics, keyed residual corrections, nearby-day estimates, and nonnegative stacking. Compact statements are expanded into ordinary Python statements. Data paths now use the local `data` folder.

This is a linear-model-based pipeline rather than a single plain regression. Training and validation can take time because the pipeline fits many models.

## 1. Load data and libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
DATA_DIR = Path('data')

def score(y, prediction):
    return {
        'RMSE': np.sqrt(mean_squared_error(y, prediction)),
        'MAE': mean_absolute_error(y, prediction),
        'R2': r2_score(y, prediction),
    }

def write_submission(test, prediction, filename):
    prediction = np.asarray(prediction)
    assert prediction.shape == (len(test),)
    assert np.isfinite(prediction).all() and (prediction > 0).all()
    result = pd.DataFrame({'id': test['id'].to_numpy(), 'price': prediction})
    assert result['id'].is_unique
    result.to_csv(DATA_DIR / filename, index=False)
    print('Saved:', DATA_DIR / filename)
    return result

In [2]:
from collections import defaultdict
from scipy.optimize import nnls
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
print('Train:', train.shape, '| Test:', test.shape)
display(pd.DataFrame({'train_missing': train.isna().sum(), 'test_missing': test.isna().sum()}))

Train: (15589, 12) | Test: (3898, 11)


,train_missing,test_missing
airline,3089,799.0
arrival_time,0,0.0
days_left,0,0.0
departure_time,0,0.0
destination_city,0,0.0
duration,3095,776.0
flight,0,0.0
flight_class,0,0.0
id,0,0.0
price,0,NaN


## 2. Prepare flight features
Use carrier prefix, route/stops duration medians, a missing-duration flag, log days left, and a last-three-days indicator.

In [4]:
def make_features(df, ref):
    X = df.copy()
    X['code'] = X.flight.str.split().str[0]
    X['dur_missing'] = X.duration.isna().astype(int)
    key = ['source_city', 'destination_city', 'stops']
    med = ref.groupby(key).duration.median().rename('dur_grp').reset_index()
    X = X.merge(med, on=key, how='left')
    X['duration'] = X.duration.fillna(X.dur_grp).fillna(ref.duration.median())
    X['dl_log'] = np.log1p(X.days_left)
    X['last3'] = (X.days_left <= 3).astype(int)
    return X
CATS = ['code', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city']
NUMS = ['duration', 'days_left', 'dl_log', 'last3', 'dur_missing']

## 3. Fit the four Ridge variants
Ridge learns weighted relationships with a penalty to limit extreme coefficients. Categories are one-hot encoded; numerical inputs and pairwise interactions are scaled. Separate models handle Economy and two carrier-based Business groups.

Fare statistics summarize similar training tickets. Group statistics exclude the current training row where peers exist; the final segment-wide fallback uses all segment prices. Small Ridge models then correct remaining errors by flight and itinerary keys. Four predictions are averaged. These steps are retained because removing them would change the original model.

In [5]:
from collections import defaultdict
DOMESTIC = {'6E', 'AI', 'IX'}
BASE_ALPHA = {'Economy': 100, 'BizDom': 500, 'BizInt': 500}
LEVELS = ['k_it', 'k_fl_r', 'k_fl', 'k_car_r']
STAT_NAMES = ['loo_med', 'loo_q25', 'loo_min', 'loo_mean']

def add_keys(X):
    dl = X.days_left.astype(str)
    X['k_it'] = X.flight + '|' + X.source_city + '|' + X.destination_city + '|' + X.departure_time + '|' + X.arrival_time + '|' + X.stops + '|' + X.flight_class
    X['k_fl_r'] = X.flight + '|' + X.source_city + '|' + X.destination_city + '|' + X.flight_class
    X['k_fl'] = X.flight + '|' + X.flight_class
    X['k_car_r'] = X.code + '|' + X.source_city + '|' + X.destination_city + '|' + X.stops + '|' + X.flight_class
    X['k_cs'] = X.code + '|' + X.stops + '|' + X.flight_class + '|' + dl
    X['k_f_dl'] = X.flight + '|' + X.flight_class + '|' + dl
    X['k_cr'] = X.code + '|' + X.source_city + '|' + X.destination_city + '|' + X.stops + '|' + X.flight_class + '|' + dl
    X['k_fr'] = X.flight + '|' + X.source_city + '|' + X.destination_city + '|' + X.flight_class + '|' + dl
    X['k_full'] = X.k_it + '|' + dl
    return X

def price_stats(A, y_a, B, leave_one_out):
    tables = []
    for lv in LEVELS:
        t = defaultdict(list)
        for i, (k, p) in enumerate(zip(A[lv].values, y_a)):
            t[k].append((i, p))
        tables.append(t)
    out = np.zeros((len(B), 4))
    for r in range(len(B)):
        for lv, t in zip(LEVELS, tables):
            items = t.get(B[lv].values[r], [])
            if leave_one_out:
                items = [(i, p) for i, p in items if i != r]
            if items:
                v = np.array([p for _, p in items])
                out[r] = (np.median(v), np.percentile(v, 25), v.min(), v.mean())
                break
        else:
            out[r] = (np.median(y_a), np.percentile(y_a, 25), y_a.min(), y_a.mean())
    return out
S_BASE = [('flight', 5), ('k_f_dl', 2), ('k_cr', 3), ('k_fr', 1), ('k_full', 0.1)]
VARIANTS = [dict(stats=['loo_med', 'loo_q25'], mult=5, stages=S_BASE), dict(stats=STAT_NAMES, mult=3, stages=S_BASE), dict(stats=['loo_med', 'loo_q25'], mult=10, stages=[('flight', 5), ('k_cs', 3)] + S_BASE[1:]), dict(stats=[], mult=1, stages=S_BASE)]

def segment(X):
    return np.where(X.flight_class == 'Economy', 'Economy', np.where(X.code.isin(DOMESTIC), 'BizDom', 'BizInt'))

def fit_predict(train, apply_to):
    Xtr = add_keys(make_features(train, train))
    Xap = add_keys(make_features(apply_to, train))
    str_, sap = (segment(Xtr), segment(Xap))
    pred = np.zeros(len(apply_to))
    for seg in np.unique(str_):
        itr, iap = (str_ == seg, sap == seg)
        if iap.sum() == 0:
            continue
        A, B = (Xtr[itr].reset_index(drop=True), Xap[iap].reset_index(drop=True))
        y = train.price.values[itr]
        sa, sb = (price_stats(A, y, A, True), price_stats(A, y, B, False))
        for j, nm in enumerate(STAT_NAMES):
            A[nm], B[nm] = (sa[:, j], sb[:, j])
        preds = []
        for v in VARIANTS:
            pre = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATS), ('n', StandardScaler(), NUMS + v['stats'])])
            stage1 = Pipeline([('pre', pre), ('poly', PolynomialFeatures(2, interaction_only=True, include_bias=False)), ('scale', StandardScaler()), ('ridge', Ridge(alpha=BASE_ALPHA[seg] * v['mult']))]).fit(A, y)
            p, resid = (stage1.predict(B), y - stage1.predict(A))
            for col, alpha in v['stages']:
                m = Pipeline([('oh', ColumnTransformer([('k', OneHotEncoder(handle_unknown='ignore'), [col])])), ('ridge', Ridge(alpha=alpha))]).fit(A, resid)
                p = p + m.predict(B)
                resid = resid - m.predict(A)
            preds.append(np.clip(p, y.min(), y.max() * 1.1))
        pred[np.where(iap)[0]] = np.mean(preds, axis=0)
    return pred

## 4. Combine with nearby-day fare estimates
Tickets with closer `days_left` receive higher weight. Nonnegative least squares combines the Ridge estimate with itinerary and carrier/route estimates, separately by segment and peer availability.

In [6]:
from sklearn.model_selection import KFold
from scipy.optimize import nnls
TAU = 5.0

def kernel_pred(Atr, y, Bap, key):
    tab = defaultdict(list)
    for k, dl, p in zip(Atr[key].values, Atr.days_left.values, y):
        tab[k].append((dl, p))
    tab = {k: np.array(v) for k, v in tab.items()}
    out = np.full(len(Bap), np.nan)
    for r, (k, dl) in enumerate(zip(Bap[key].values, Bap.days_left.values)):
        arr = tab.get(k)
        if arr is not None:
            w = np.exp(-np.abs(arr[:, 0] - dl) / TAU)
            out[r] = (w * arr[:, 1]).sum() / w.sum()
    return out

def stack_inputs(train, apply_to):
    P = fit_predict(train, apply_to)
    Xtr = add_keys(make_features(train, train))
    Xap = add_keys(make_features(apply_to, train))
    Ki = kernel_pred(Xtr, train.price.values, Xap, 'k_it')
    Kc = kernel_pred(Xtr, train.price.values, Xap, 'k_car_r')
    peer = ~np.isnan(Ki)
    Kc = np.where(np.isnan(Kc), P, Kc)
    Ki = np.where(np.isnan(Ki), Kc, Ki)
    return dict(X=np.column_stack([P, Ki, Kc]), seg=segment(Xap), peer=peer)
GROUPS = [(s, p) for s in ['Economy', 'BizDom', 'BizInt'] for p in (True, False)]

def oof_inputs(df, n_splits=5, seed=42):
    n = len(df)
    X = np.zeros((n, 3))
    seg = np.empty(n, dtype=object)
    peer = np.zeros(n, bool)
    fold = np.zeros(n, int)
    for f, (tr, va) in enumerate(KFold(n_splits, shuffle=True, random_state=seed).split(df)):
        r = stack_inputs(df.iloc[tr], df.iloc[va])
        X[va], seg[va], peer[va], fold[va] = (r['X'], r['seg'], r['peer'], f)
    return dict(X=X, seg=seg, peer=peer, fold=fold)

def fit_weights(inp, y, mask=None):
    mask = np.ones(len(y), bool) if mask is None else mask
    W = {}
    for s, p in GROUPS:
        m = mask & (inp['seg'] == s) & (inp['peer'] == p)
        W[s, p] = nnls(inp['X'][m], y[m])[0] if m.sum() >= 30 else np.array([1.0, 0.0, 0.0])
    return W

def apply_weights(inp, W):
    out = np.zeros(len(inp['X']))
    for s, p in GROUPS:
        m = (inp['seg'] == s) & (inp['peer'] == p)
        out[m] = inp['X'][m] @ W[s, p]
    return out

In [7]:
def predict_lr(training, application):
    # Learn combination weights from training-only OOF predictions.
    training = training.reset_index(drop=True)
    application = application.reset_index(drop=True)
    inputs = oof_inputs(training)
    weights = fit_weights(inputs, training['price'].to_numpy())
    return apply_weights(stack_inputs(training, application), weights)

## 5. Development cross-validation
This reproduces the original OOF calculation. It is not fully nested cross-validation: predictions used to fit stack weights on other folds can come from base models that saw the held-out fold. Use the separate holdout below as an additional check.

In [8]:
y_all = train.price.values
oof = oof_inputs(train)
mse_model = mean_squared_error(y_all, oof['X'][:, 0])
stacked = np.zeros(len(y_all))
for f in range(5):
    W_f = fit_weights(oof, y_all, mask=oof['fold'] != f)
    sel = oof['fold'] == f
    stacked[sel] = apply_weights({k: v[sel] for k, v in oof.items()}, W_f)
mse_stack = mean_squared_error(y_all, stacked)
print(f'Model only (4-variant blend): MSE {mse_model:,.0f}  RMSE {np.sqrt(mse_model):,.0f}  R2 {1 - mse_model / y_all.var():.4f}')
print(f'With linear stack           : MSE {mse_stack:,.0f}  RMSE {np.sqrt(mse_stack):,.0f}  R2 {1 - mse_stack / y_all.var():.4f}')
W_final = fit_weights(oof, y_all)
for k, w in W_final.items():
    print(k, 'weights [model, kernel(itinerary), kernel(carrier+route)] =', np.round(w, 2))

c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.002854045924549671.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0074110192658631115.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.00022283173854896628.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.493502882187581e-05.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shu

Model only (4-variant blend): MSE 96,694,922  RMSE 9,833  R2 0.9368
With linear stack           : MSE 94,015,215  RMSE 9,696  R2 0.9386
('Economy', True) weights [model, kernel(itinerary), kernel(carrier+route)] = [0.73 0.   0.27]
('Economy', False) weights [model, kernel(itinerary), kernel(carrier+route)] = [0.09 0.93 0.  ]
('BizDom', True) weights [model, kernel(itinerary), kernel(carrier+route)] = [0.55 0.4  0.04]
('BizDom', False) weights [model, kernel(itinerary), kernel(carrier+route)] = [0.02 1.02 0.  ]
('BizInt', True) weights [model, kernel(itinerary), kernel(carrier+route)] = [0.83 0.09 0.08]
('BizInt', False) weights [model, kernel(itinerary), kernel(carrier+route)] = [0.97 0.   0.  ]


## 6. Separate holdout evaluation

In [9]:
fit_rows, valid_rows = train_test_split(train, test_size=0.2, random_state=SEED)
validation_predictions = predict_lr(fit_rows, valid_rows.drop(columns='price'))
display(pd.DataFrame([score(valid_rows['price'], validation_predictions)]).round(4))

c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0035189208872747277.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0007283231099583185.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:264: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0035467921848659385.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.000643658957376238.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\s

,RMSE,MAE,R2
0,9362.1582,3198.7008,0.9378


## 7. Fit on all training data and save
Reuse the full-training OOF weights computed in Section 5.

In [10]:
test_predictions = apply_weights(stack_inputs(train, test), W_final)
submission = write_submission(test, test_predictions, 'submission_lr_simple.csv')
display(submission.head())

c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0016528596044158477.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\shubh\.conda\envs\idk_ml\Lib\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.886008815798946e-05.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Saved: data\submission_lr_simple.csv


,id,price
0,1,7938.913219
1,2,7452.108784
2,3,28953.263795
3,4,32576.539848
4,5,4421.738585
